# Marketing Spend Data Cleaning

This notebook prepares the CRM **marketing spend** table for campaign and acquisition analysis.

### Main tasks
- standardize column names
- remove exact duplicate rows
- inspect numerical consistency across impressions, clicks, and spend
- distinguish non-paid / partially tracked traffic from invalid records
- normalize categorical fields and handle missing campaign metadata
- save a clean dataset for downstream marketing analysis

> **Data note:** the original CRM files are not included in the public repository.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from helpers import to_snake, df_overview, df_clean_summary

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and Initial Inspection

In [ ]:
spend = pd.read_excel(RAW_DIR / 'Spend (Done).xlsx')

In [ ]:
# Standardize column names and parse the activity date
spend.columns = [to_snake(c) for c in spend.columns]
spend['date'] = pd.to_datetime(spend['date'], errors='coerce')

In [ ]:
# Display a compact data-quality overview
df_overview(spend)

In [ ]:
# Inspect the main categorical fields
for col in ['source', 'campaign']:
    print(f"\nUnique values in '{col}':")
    print(spend[col].unique())

## 2. Duplicate Rows

In [ ]:
# Remove exact duplicate rows
print(f'Exact duplicates: {spend.duplicated().sum()}')
spend = spend.drop_duplicates()
print(f'Rows after deduplication: {len(spend)}')

## 3. Numerical Consistency Checks

In [ ]:
cols_to_check = ['impressions', 'spend', 'clicks']

### 3.1. Negative Values

In [ ]:
print('Negative values:')
print((spend[cols_to_check] < 0).sum())

### 3.2. Clicks Present While Spend Is Zero

In [ ]:
zero_spend_clicks = (spend['spend'] == 0) & (spend['clicks'] > 0)
print('Rows with spend=0 and clicks>0:', zero_spend_clicks.sum())

These records are plausible for non-paid or non-tracked acquisition channels, so they are retained.

In [ ]:
spend.loc[zero_spend_clicks, 'source'].value_counts()

### 3.3. Fully Zero Activity Rows

In [ ]:
all_zero = (
    (spend['impressions'] == 0)
    & (spend['clicks'] == 0)
    & (spend['spend'] == 0)
)
print('Rows with zero values across all three metrics:', all_zero.sum())

These rows can represent campaign/source-day combinations with no recorded activity. They are retained rather than treated as errors.

In [ ]:
spend[all_zero]['source'].value_counts()

### 3.4. Impressions Equal Zero While Clicks Are Present

In [ ]:
zero_impressions_clicks = (
    (spend['impressions'] == 0)
    & (spend['clicks'] > 0)
)
print('Rows with impressions=0 and clicks>0:', zero_impressions_clicks.sum())

In [ ]:
spend.loc[zero_impressions_clicks, 'source'].value_counts()

In [ ]:
spend.loc[zero_impressions_clicks].groupby('source')[['spend', 'clicks']].agg(['sum', 'mean'])

Most affected sources are non-paid or partially tracked channels such as Organic, CRM, Partnership, Offline, Bloggers, and Telegram.

SMM is an exception: spend is recorded while impressions are zero. This is treated as a tracking limitation rather than automatically removing the records.

### 3.5. Spend Is Positive While Impressions Are Zero

In [ ]:
paid_without_impressions = (
    (spend['spend'] > 0)
    & (spend['impressions'] == 0)
)
print('Rows with spend>0 and impressions=0:', paid_without_impressions.sum())

In [ ]:
spend[paid_without_impressions]['source'].value_counts()

In [ ]:
spend.groupby(by='source')[['spend', 'clicks']].sum()

Some paid channels contain spend and clicks even when impressions are not recorded. These rows are retained because the pattern may reflect platform-specific tracking limitations.

## 4. Categorical Fields: Normalization and Missing Values

`campaign`, `ad_group`, and `ad` contain substantial missing metadata. Because the missing values cannot be reconstructed reliably from the available CRM data, they are labeled as `Unknown`.

In [ ]:
category_cols = ['source', 'campaign', 'ad_group', 'ad']
fill_cols = ['campaign', 'ad_group', 'ad']

# Normalize string values
for col in category_cols:
    spend[col] = spend[col].str.strip()

# Label missing campaign metadata that cannot be reconstructed reliably
for col in fill_cols:
    missing_count = spend[col].isna().sum()
    spend[col] = spend[col].fillna('Unknown')
    print(f'{col}: filled {missing_count} missing values with "Unknown"')

## 5. Final Quality Check

In [ ]:
df_clean_summary(spend)

## 6. Save Processed Data

In [ ]:
# Pickle preserves parsed data types for downstream notebooks
spend.to_pickle(PROCESSED_DIR / 'spend_clean.pkl')

print('Saved: spend_clean.pkl')
print(f'Shape: {spend.shape}')

## 7. Output Dataset

| Column | Type | Description |
|---|---|---|
| `date` | `datetime64` | Date of the recorded marketing activity |
| `source` | `object` | Acquisition / advertising source |
| `campaign` | `object` | Campaign name; `Unknown` when unavailable |
| `impressions` | `int64` | Number of ad impressions |
| `spend` | `float64` | Marketing spend |
| `clicks` | `int64` | Number of clicks |
| `ad_group` | `object` | Ad group; `Unknown` when unavailable |
| `ad` | `object` | Ad / creative; `Unknown` when unavailable |

**Cleaned dataset size:** 19,862 rows and 8 columns.

## 8. Cleaning Summary

**Source table:** marketing spend data with 8 fields: `date`, `source`, `campaign`, `impressions`, `spend`, `clicks`, `ad_group`, and `ad`.

| Step | Action | Result |
|---|---|---|
| 1 | Column standardization | Converted column names to `snake_case` |
| 2 | Deduplication | Removed exact duplicate rows |
| 3 | Numerical validation | Checked `impressions`, `spend`, and `clicks` for negative values |
| 4 | Tracking-anomaly review | Investigated zero-spend / positive-click and zero-impression patterns instead of deleting plausible records |
| 5 | Categorical normalization | Trimmed whitespace in source and campaign metadata |
| 6 | Missing metadata | Filled unavailable `campaign`, `ad_group`, and `ad` values with `Unknown` |

### Output
- `spend_clean.pkl` — cleaned marketing spend dataset
